<h2>LLM</h2>

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

/opt/homebrew/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:


# 1. Load a small model (suitable for laptops/CPU)
model_name = "gpt2" # A classic small Transformer
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# 2. The Request (The Prompt)
prompt = "The capital of France is"
input_ids = tokenizer.encode(prompt, return_tensors="pt")

print(f"Starting Prompt: {prompt}")
print("-" * 30)

# 3. The "Inference Loop" (Generating 5 words)
for _ in range(5):
    # Get the "Logits" (Raw scores for the whole vocabulary)
    with torch.no_grad():
        outputs = model(input_ids)
        next_token_logits = outputs.logits[:, -1, :]

    # Perform "Greedy Search" (Pick the #1 highest probability)
    next_token_id = torch.argmax(next_token_logits, dim=-1).unsqueeze(0)
    
    # Add the new token to our sequence
    input_ids = torch.cat([input_ids, next_token_id], dim=-1)
    
    # Decode just the new token to show the progress
    next_word = tokenizer.decode(next_token_id[0])
    print(f"Model predicted: '{next_word}'")

# Final Result
print("-" * 30)
print(f"Final Output: {tokenizer.decode(input_ids[0])}")    #around 20 seconds

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 17989.31it/s]


Starting Prompt: The capital of France is
------------------------------
Model predicted: ' the'
Model predicted: ' capital'
Model predicted: ' of'
Model predicted: ' the'
Model predicted: ' French'
------------------------------
Final Output: The capital of France is the capital of the French


In [ ]:
#optimize
# Replace the "Inference Loop" with this "Pro" version:
# output_ids = model.generate(
#     input_ids, 
#     max_length=20, 
#     num_beams=5,          # Checks multiple paths to find the best logic
#     no_repeat_ngram_size=2, # STOPS the "capital of... capital of" loop
#     early_stopping=True
# )

# print(f"Professional Output: {tokenizer.decode(output_ids[0], skip_special_tokens=True)}")

<h2>Clean the data</h2>

In [ ]:
from transformers import pipeline
import pandas as pd

# 1. Initialize a "Task-Specific" Pipeline (Zero-Shot Classification)
# This allows us to classify text without any training.
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# 2. Our "Messy" Data (Could be from a CSV or Web Scraper)
reviews = [
    "I bought the Galaxy S24 yesterday. The screen is beautiful but the battery dies in 2 hours. Very disappointed.",
    "The new MacBook is so fast! I love the keyboard, though it was quite expensive.",
    "Avoid this blender. It started smoking the first time I made a smoothie. Dangerous!"
]

# 3. Define the "Labels" we want the AI to look for
candidate_labels = ["electronics", "kitchen", "positive", "negative", "safety-hazard"]

# 4. Process the data
results = []
for text in reviews:
    prediction = classifier(text, candidate_labels)
    # Get the top label and its score
    top_label = prediction['labels'][0]
    score = prediction['scores'][0]
    
    results.append({
        "Review": text[:30] + "...", 
        "Category/Sentiment": top_label,
        "Confidence": f"{score:.2%}"
    })

# 5. Output as a clean Table
df = pd.DataFrame(results)
print(df)   #30 secs

/opt/homebrew/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 515/515 [00:00<00:00, 59115.12it/s]


                              Review Category/Sentiment Confidence
0  I bought the Galaxy S24 yester...           negative     77.51%
1  The new MacBook is so fast! I ...        electronics     51.61%
2  Avoid this blender. It started...      safety-hazard     65.77%


In [ ]:
import torch

# 2. Load the model using the GPU (device_map="auto")
# We use 'phi-3.5-mini' because it is one of the best for logical Q&A
model_id = "microsoft/Phi-3.5-mini-instruct"

pipe = pipeline(
    "text-generation", 
    model=model_id, 
    device_map="auto", 
    torch_dtype=torch.bfloat16, # Uses less memory without losing quality
    trust_remote_code=True
)

# 3. Create a better prompt structure
def ask_phi(question):
    # This specific format helps the model understand it's a chat
    messages = [
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": question},
    ]
    
    # We use 'max_new_tokens' so it doesn't get cut off
    output = pipe(messages, max_new_tokens=200, temperature=0.7)
    return output[0]['generated_text'][-1]['content']

# 4. Test it
print(f"Answer: {ask_phi('Explain why the sky is blue like I am 5 years old.')}")